# Test Retrieval Consistency

This notebook simulates a number of mixed pixels from the reference library, unmixes them using multiple EO sensors, and compares the consistency of the retrieval between instruments.

In [ ]:
# packages
import os
import numpy as np
from numpy.random import default_rng
import spectral
import earthlib
from earthlib import library
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
from pysptools.abundance_maps.amaps import FCLS
from sklearn import metrics

plots = os.path.join(
    os.path.dirname(os.getcwd()), 
    'docs',
    'img'
)

In [ ]:
# simulations parameters
n_unmixing_iterations = 30
n_mixed_endmembers = 3_000
n_pure_endmembers = 300

In [ ]:
# functions

def sample_library(n):
    """Get 3-class endmember samples"""
    bare = library.subsample(n=n, by_type = "bare")
    pv = library.subsample(n=n, by_type = "vegetation")
    burn = library.subsample(n=n, by_type = "burned")

    return bare, pv, burn

def library_to_list(libraries, sensor):
    sensor = earthlib.sensors.supported_sensors[sensor]
    bare_lib, pv_lib, burn_lib = libraries

    bare_lib = bare_lib.to_sensor(sensor)
    pv_lib = pv_lib.to_sensor(sensor)
    burn_lib = burn_lib.to_sensor(sensor)

    bare = [bare_lib.data[idx] for idx in range(len(bare_lib))]
    pv = [pv_lib.data[idx] for idx in range(len(pv_lib))]
    burn = [burn_lib.data[idx] for idx in range(len(burn_lib))]
    
    return bare, pv, burn

def unmix(ref, sim, shade_normalize = True, weighted = True):
    """Run spectral unmixing on a series of simulated spectra"""
    # unpack the input data
    bare_ref, green_ref, burn_ref = ref
    bare_sim, green_sim, burn_sim = sim

    # get the size of the analysis
    n_ref = len(bare_ref)
    n_sim = len(bare_sim)

    # create the reference shade endmember
    shade_ref = np.zeros_like(bare_sim[0])

    # create output arrays
    estimated_fractions = np.zeros((n_ref, n_sim, 3))
    rmses = np.zeros((n_ref, n_sim))
    
    # set up the endmember fractions
    seed_value = 1959
    rng = default_rng(seed_value)
    endmember_fractions = rng.dirichlet(np.ones(3),size=n_sim)
    shade_fraction = 1 - rng.uniform(0, 0.5, size=n_sim)

    # create the mixed spectra
    bare_spec = bare_sim * endmember_fractions[:, 0:1]
    green_spec = green_sim * endmember_fractions[:, 1:2] * shade_fraction[..., np.newaxis]
    burn_spec = burn_sim * endmember_fractions[:, 2:]
    simulated = bare_spec + green_spec + burn_spec

    # loop over each reference combination and unmix the simulated spectra
    for idx in tqdm(range(n_ref), leave=False):
        
        reference = np.stack([
            bare_ref[idx],
            green_ref[idx],
            burn_ref[idx],
        ])

        if shade_normalize:
            reference = np.vstack((reference, shade_ref))
        
        unmixed = FCLS(simulated, reference)

        # return shade normalized fraction
        if shade_normalize:
            valid_fraction = unmixed[:, :-1]
            shade_fraction = 1 - unmixed[:, -1:]
            unmixed = np.divide(
                valid_fraction,
                shade_fraction,
                where=shade_fraction > 0,
            )
        estimated_fractions[idx] = unmixed

        # compute the spectral rmse based on agreement between modeled and observed spectra
        modeled_spectra = estimate_spectra(reference[:3], unmixed)
        rmse = spectral_rmse(simulated, modeled_spectra)
        rmses[idx] = rmse

    # compute the average across these simulations
    if weighted:
        weights = estimate_weights(rmses)
        estimated_fractions = np.average(estimated_fractions, weights=weights, axis=0)
    else:
        estimated_fractions = estimated_fractions.mean(axis=0)

    return simulated, endmember_fractions, estimated_fractions

def estimate_spectra(endmembers, fractions):
    """Model the spectral response best estimated by the unmixing fractions"""
    n_endmembers, n_channels = endmembers.shape
    n_fractions = len(fractions)
    modeled_fractions = np.zeros((n_endmembers, n_fractions, n_channels), dtype=np.float32)
    for idx in range(n_endmembers):
        modeled_fractions[idx] = endmembers[idx:idx+1] * fractions[..., idx:idx+1]
    modeled = modeled_fractions.sum(axis=0)
    return modeled

def spectral_rmse(reference, modeled):
    """Estimate the RMSE of the estimated spectral response"""
    residuals = reference - modeled
    rmse = np.sqrt((residuals ** 2).mean(axis=1))
    return rmse

def estimate_weights(rmse):
    """Apply inverse weights where lower RMSE scores are weighted higher"""
    # here, rmse values lower than the mean have weights > 1
    # rmse values greater than average have weights < 1
    avg = rmse.mean()
    weights = avg / rmse

    # apply uniform weights across all channels
    weights = weights[..., np.newaxis].repeat(3, axis=-1)

    return weights

def plot_retrieval(simulated, estimated, sensor):
    """Create 2d Histograms with simulated and estimated endmember fractions"""

    # setup
    labels = ["Bare ground", "Green veg.", "Burned area"]
    rng = (0, 1)
    ticks = (0, 0.25, 0.5, 0.75, 1.0)
    tick_labels = (0, 25, 50, 75, 100)
    
    # array access
    bare_sim, bare_est = simulated[:, 0], estimated[:, 0]
    green_sim, green_est = simulated[:, 1], estimated[:, 1]
    burn_sim, burn_est = simulated[:, 2], estimated[:, 2]

    # metric calculation
    bare_r2 = metrics.r2_score(bare_sim, bare_est)
    green_r2 = metrics.r2_score(green_sim, green_est)
    burn_r2 = metrics.r2_score(burn_sim, burn_est)
    bare_mae = metrics.mean_absolute_error(bare_sim, bare_est)
    green_mae = metrics.mean_absolute_error(green_sim, green_est)
    burn_mae = metrics.mean_absolute_error(burn_sim, burn_est)

    # array creation for simple zip/unzipping
    sims = (bare_sim, green_sim, burn_sim)
    ests = (bare_est, green_est, burn_est)
    r2s = (bare_r2, green_r2, burn_r2)
    maes = (bare_mae, green_mae, burn_mae)

    # plotting
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(9, 3.75), dpi=125, sharex=True, sharey=True)
    for sim, est, r2, mae, label, ax in zip(sims, ests, r2s, maes, labels, axs):
        ax.hist2d(
            sim,
            est,
            bins=20,
            range=(rng, rng),
            cmin=1,
            cmap="GnBu",
            norm=matplotlib.colors.LogNorm()
        )
        ax.plot(
            rng,
            rng,
            label=f"$r^2: {r2:0.2f}$\n$MAE: {mae:0.2f}$",
            color='black',
            linestyle='--',
        )
        ax.set_title(label, fontsize='medium')
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        ax.set_xticks(ticks, tick_labels)
        ax.set_yticks(ticks, tick_labels)
        ax.legend()

    plt.suptitle(f"{sensor}", fontsize='large')
    fig.supxlabel('Simulated Fraction (%)')
    fig.supylabel('Estimated Fraction (%)')
    plt.tight_layout()
    plt.savefig(f'{plots}/{sensor}-unmixing-retrieval.png', dpi=200)

def model_performance_metrics(simulated, estimated):
    """Compute R2 and MAE scores"""
    # array access
    bare_sim, bare_est = simulated[:, 0], estimated[:, 0]
    green_sim, green_est = simulated[:, 1], estimated[:, 1]
    burn_sim, burn_est = simulated[:, 2], estimated[:, 2]

    # metric calculation
    bare_r2 = metrics.r2_score(bare_sim, bare_est)
    green_r2 = metrics.r2_score(green_sim, green_est)
    burn_r2 = metrics.r2_score(burn_sim, burn_est)
    bare_mae = metrics.mean_absolute_error(bare_sim, bare_est)
    green_mae = metrics.mean_absolute_error(green_sim, green_est)
    burn_mae = metrics.mean_absolute_error(burn_sim, burn_est)

    return bare_r2, green_r2, burn_r2, bare_mae, green_mae, dry_mae

We'll first sample a base set of "reference" spectra to use for running the unmixing. Then we'll subsample a separate set of "simulated" spectra that we'll use to mix and create synthetic observations.

We'll generate these data upfront, then spectrally resample them to the wavelenghts of different instruments we're going to compare.

In [ ]:
# spectral sampling
reference_libraries = sample_library(n=n_unmixing_iterations)
simulated_libraries = sample_library(n=n_mixed_endmembers)

# convert to sensor-specific arrays
modis_ref = library_to_list(reference_libraries, 'MODIS')
modis_sim = library_to_list(simulated_libraries, 'MODIS')

landsat_ref = library_to_list(reference_libraries, 'Landsat8')
landsat_sim = library_to_list(simulated_libraries, 'Landsat8')

sentinel_ref = library_to_list(reference_libraries, 'Sentinel2')
sentinel_sim = library_to_list(simulated_libraries, 'Sentinel2')

Next we'll unmix the simulated endmember fractions, then plot the relative retrieval precision.

These scatter plots show how well the unmixed fractions match the simulated mixture fractions.

In [ ]:
# MODIS retrieval
modis_reflectance, modis_simulated, modis_estimated = unmix(modis_ref, modis_sim, weighted=True)
plot_retrieval(modis_simulated, modis_estimated, "MODIS")

# Landsat retrieval
landsat_reflectance, landsat_simulated, landsat_estimated = unmix(landsat_ref, landsat_sim, weighted=True)
plot_retrieval(landsat_simulated, landsat_estimated, "Landsat")

# Sentinel2 retrieval
sentinel_reflectance, sentinel_simulated, sentinel_estimated = unmix(sentinel_ref, sentinel_sim, weighted=True)
plot_retrieval(sentinel_simulated, sentinel_estimated, "Sentinel2")

Next, we'll create a 3x3 panel correlation plot figure.

Top row: BARE GROUND Landsat/MODIS, Landsat/Sentinel2, MODIS/Sentinel2
Mid row: VEGETATION  Landsat/MODIS, Landsat/Sentinel2, MODIS/Sentinel2
Bot row: BURNED Landsat/MODIS, Landsat/Sentinel2, MODIS/Sentinel2

In [ ]:
def plot_correlations(ax, x, y, xlabel, ylabel):
    r2 = metrics.r2_score(x, y)
    mae = metrics.mean_absolute_error(x, y)
    rmse = metrics.root_mean_squared_error(x, y)
    rng = (0, 1.0)
    ticks = (0, 0.25, 0.5, 0.75, 1.0)
    tick_labels = (0, 25, 50, 75, 100)
    
    ax.scatter(
        x,
        y,
        color='#009db1',
        alpha=0.2,
        s=10,
    )
    ax.plot(
        rng,
        rng,
        label=f"$r^2: {r2:0.3f}$\n$MAE: {mae:0.2f}$\n$RMSE: {rmse:0.2f}$",
        color='black',
        linestyle='--',
    )

    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(ticks, tick_labels)
    ax.set_yticks(ticks, tick_labels)
    ax.legend(loc='upper left')

# labels
band_labels = ["Bare ground (%)", "Green vegetation (%)", "Burned area (%)"]

# figure creation
fig = plt.figure(constrained_layout=True, dpi=125, figsize=(9,9.5))
fig.suptitle('Cross-sensor land cover fraction estimation consistency', fontsize=14)
subfigs = fig.subfigures(nrows=3, ncols=1)
for row, subfig in enumerate(subfigs):
    subfig.suptitle(band_labels[row])
    axs = subfig.subplots(nrows=1, ncols=3)

    # landsat/modis, landsat/s2, modis/s2
    plot_correlations(axs[0], modis_estimated[:, row], landsat_estimated[:, row], "MODIS", "Landsat")
    plot_correlations(axs[1], sentinel_estimated[:, row], landsat_estimated[:, row], "Sentinel-2", "Landsat")
    plot_correlations(axs[2], sentinel_estimated[:, row], modis_estimated[:, row], "Sentinel-2", "MODIS")

plt.savefig(f"{plots}/multi-sensor-retrieval-consistency.png", dpi=200)

Run this same comparison, but with the raw surface reflectance data to compare the retrieval consistency at a band-level.

In [ ]:
# get band indices for reflectance data
sentinel_nir = earthlib.sensors.Sentinel2.band_descriptions.index('near infrared')
sentinel_swir1 = earthlib.sensors.Sentinel2.band_descriptions.index('shortwave infrared 1')
sentinel_swir2 = earthlib.sensors.Sentinel2.band_descriptions.index('shortwave infrared 2')

landsat_nir = earthlib.sensors.Landsat8.band_descriptions.index('near infrared')
landsat_swir1 = earthlib.sensors.Landsat8.band_descriptions.index('shortwave infrared 1')
landsat_swir2 = earthlib.sensors.Landsat8.band_descriptions.index('shortwave infrared 2')

modis_nir = earthlib.sensors.MODIS.band_descriptions.index('near infrared')
modis_swir1 = earthlib.sensors.MODIS.band_descriptions.index('shortwave infrared 1')
modis_swir2 = earthlib.sensors.MODIS.band_descriptions.index('shortwave infrared 2')

# figure labels
band_labels = ["NIR reflectance (%)", "SWIR-1 reflectance (%)", "SWIR-2 reflectance (%)"]

bands = [
    [sentinel_nir, landsat_nir, modis_nir],
    [sentinel_swir1, landsat_swir1, modis_swir1],
    [sentinel_swir2, landsat_swir2, modis_swir2],
]

# figure creation
fig = plt.figure(constrained_layout=True, dpi=125, figsize=(9,9.5))
fig.suptitle('Cross-sensor reflectance consistency', fontsize=14)
subfigs = fig.subfigures(nrows=3, ncols=1)
for row, subfig in enumerate(subfigs):
    subfig.suptitle(band_labels[row])
    axs = subfig.subplots(nrows=1, ncols=3)

    # band indexing
    sentinel_band, landsat_band, modis_band = bands[row]

    # landsat/modis, landsat/s2, modis/s2
    plot_correlations(axs[0], modis_reflectance[:, modis_band], landsat_reflectance[:, landsat_band], "MODIS", "Landsat")
    plot_correlations(axs[1], sentinel_reflectance[:, sentinel_band], landsat_reflectance[:, landsat_band], "Sentinel-2", "Landsat")
    plot_correlations(axs[2], sentinel_reflectance[:, sentinel_band], modis_reflectance[:, modis_band], "Sentinel-2", "MODIS")

# plt.savefig(f"{plots}/multi-sensor-reflectance-consistency.png", dpi=200)